In [ ]:
# %%
import seaborn as sns
import geopandas as gpd
import networkx as nx
import matplotlib.pyplot as plt
from pathlib import Path
import os


# Working directory = notebooks/
PROJECT_ROOT = Path(os.getcwd()).resolve().parents[0]
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
VISUAL_DIR = PROJECT_ROOT / "outputs"


In [ ]:
def load_processed(name):
    return pd.read_parquet(PROCESSED_DIR / name)

# %%
# Load scoring table
scores = load_processed("notepad_03_criticality_scores.parquet")
scores = scores.set_index("Mineral Code")

# Style
sns.set_theme(style="whitegrid", font_scale=1.2)

# 5. Save ranked table for dashboard use
ranked = scores.sort_values("Rank")
ranked.to_csv(PROCESSED_DIR / "notepad_04_ranked_table.csv")

# ranked.head(10)


In [ ]:
# %%

# %%
# 2. Supply Risk vs Demand Importance scatter plot
plt.figure(figsize=(10, 8))

sns.scatterplot(
    data=scores,
    x="SupplyRisk",
    y="DemandImportance",
    hue="CriticalityScore",
    size="CriticalityScore",
    palette="viridis",
    sizes=(50, 300),
    legend="brief"
)

for mineral, row in scores.iterrows():
    plt.text(row["SupplyRisk"] + 0.005, row["DemandImportance"] + 0.005, mineral)

plt.title("Supply Risk vs Demand Importance")
plt.xlabel("Supply Risk")
plt.ylabel("Demand Importance")
plt.tight_layout()
plt.savefig(VISUAL_DIR / "supply_vs_demand.png", dpi=300, bbox_inches="tight")

plt.show()

# 2. Supply Risk vs Demand Importance scatter plot
plt.figure(figsize=(10, 8))

sns.scatterplot(
    data=scores,
    x="SupplyRisk",
    y="DemandImportance",
    hue="CriticalityScore",
    size="CriticalityScore",
    palette="viridis",
    sizes=(50, 300),
    legend="brief"
)

for mineral, row in scores.iterrows():
    plt.text(row["SupplyRisk"] + 0.005, row["DemandImportance"] + 0.005, mineral)

plt.title("Supply Risk vs Demand Importance")
plt.xlabel("Supply Risk")
plt.ylabel("Demand Importance")
plt.tight_layout()
plt.savefig(VISUAL_DIR / "supply_vs_demand.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
# 1. Criticality ranking bar chart
plt.figure(figsize=(12, 10))
ranked = scores.sort_values("CriticalityScore", ascending=False)

sns.barplot(
    data=ranked,
    x="CriticalityScore",
    y=ranked.index,
    palette="viridis"
)

plt.title("Criticality Score Ranking")
plt.xlabel("Criticality Score")
plt.ylabel("Mineral Code")
plt.tight_layout()
plt.savefig(VISUAL_DIR / "criticality_ranking.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 3. Quadrant chart (risk vs importance)
plt.figure(figsize=(10, 8))

median_supply = scores["SupplyRisk"].median()
median_demand = scores["DemandImportance"].median()

sns.scatterplot(
    data=scores,
    x="SupplyRisk",
    y="DemandImportance",
    hue="CriticalityScore",
    palette="viridis",
    s=120
)

# Quadrant lines
plt.axvline(median_supply, color="black", linestyle="--", alpha=0.7)
plt.axhline(median_demand, color="black", linestyle="--", alpha=0.7)

for mineral, row in scores.iterrows():
    plt.text(row["SupplyRisk"] + 0.005, row["DemandImportance"] + 0.005, mineral)

plt.title("Criticality Quadrants")
plt.xlabel("Supply Risk")
plt.ylabel("Demand Importance")
plt.tight_layout()
plt.savefig(VISUAL_DIR / "criticality_quadrants.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 4. Heatmap of all normalised metrics
norm_cols = [
    "HHI_norm",
    "ProducerDiversity_norm",
    "SectorUsage_norm",
    "EconomicImportance_norm",
    "SupplyRisk",
    "DemandImportance",
    "CriticalityScore"
]

plt.figure(figsize=(14, 10))
sns.heatmap(
    scores[norm_cols],
    cmap="viridis",
    linewidths=0.5
)

plt.title("Normalised Metrics Heatmap")
plt.tight_layout()
plt.savefig(VISUAL_DIR / "metrics_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Load world geometry


world = gpd.read_file(
    r"C:\Users\rej4m\OneDrive\Documents\Hobby Projects\Critical Minerals\data\ne_110m_admin_0_countries\ne_110m_admin_0_countries.shp"
)

world = world.rename(columns={"ISO_A3": "Country Code"})

# world = world.rename(columns={"ISO_A3": "Country Code"})



# Load matrices
producer_matrix = load_processed("mineral_country_matrix.parquet")
manufacturer_matrix = load_processed("sector_country_matrix.parquet")

In [ ]:
print(world.columns)

In [ ]:
producer_matrix = load_processed("mineral_country_matrix.parquet")
producer_counts = producer_matrix.sum(axis=1).rename("Mineral Production Count")

producer_geo = world.merge(producer_counts, on="Country Code", how="left")

plt.figure(figsize=(14, 8))
producer_geo.plot(
    column="Mineral Production Count",
    cmap="viridis",
    legend=True,
    missing_kwds={"color": "lightgrey"},
    edgecolor="black",
    linewidth=0.3
)

plt.title("Global Mineral Producer Countries")
plt.axis("off")

plt.savefig(VISUAL_DIR / "producer_countries_map.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
manufacturer_matrix = load_processed("sector_country_matrix.parquet")
manufacturer_counts = manufacturer_matrix.sum(axis=0).rename("Sector Presence Count")

manufacturer_geo = world.merge(manufacturer_counts, on="Country Code", how="left")

plt.figure(figsize=(14, 8))
manufacturer_geo.plot(
    column="Sector Presence Count",
    cmap="plasma",
    legend=True,
    missing_kwds={"color": "lightgrey"},
    edgecolor="black",
    linewidth=0.3
)

plt.title("Global Manufacturer Countries")
plt.axis("off")

plt.savefig(VISUAL_DIR / "manufacturer_countries_map.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:


PROJECT_ROOT = Path(os.getcwd()).resolve().parents[0]
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

def load_processed(name):
    return pd.read_parquet(PROCESSED_DIR / name)

# -----------------------------
# LOAD NOTEBOOK 3 OUTPUTS
# -----------------------------
links = load_processed("country_links.parquet")
dependency_matrix = load_processed("dependency_matrix.parquet")

# -----------------------------
# BUILD NETWORKX GRAPH
# -----------------------------
G = nx.DiGraph()

# Add nodes with attributes
for _, row in links.iterrows():
    G.add_node(
        row["Country Code"],
        country=row["Country"],
        producer_strength=row["Producer Strength"],
        manufacturer_strength=row["Manufacturer Strength"],
        capability=row["Capability Score"],
        dependency_class=row["Dependency Class"]
    )

# Add edges from dependency matrix
for producer in dependency_matrix.index:
    for manufacturer in dependency_matrix.columns:
        weight = dependency_matrix.loc[producer, manufacturer]
        if weight > 0:  # only meaningful edges
            G.add_edge(
                producer,
                manufacturer,
                weight=weight
            )

print(f"Graph built: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# -----------------------------
# STATIC NETWORK PLOT
# -----------------------------
plt.figure(figsize=(14, 10))

# Position nodes using spring layout
pos = nx.spring_layout(G, k=0.4, iterations=50)

# Node colours by dependency class
class_colors = {
    "Integrated": "gold",
    "Producer Only": "steelblue",
    "Manufacturer Only": "firebrick",
    "Neither": "grey"
}

node_colors = [
    class_colors[G.nodes[n]["dependency_class"]] for n in G.nodes()
]

# Node sizes by capability score
node_sizes = [
    800 * G.nodes[n]["capability"] for n in G.nodes()
]

# Draw nodes
nx.draw_networkx_nodes(
    G, pos,
    node_color=node_colors,
    node_size=node_sizes,
    alpha=0.9
)

# Draw edges weighted by link score
edge_weights = [G[u][v]["weight"] * 5 for u, v in G.edges()]
nx.draw_networkx_edges(
    G, pos,
    width=edge_weights,
    alpha=0.4,
    arrows=True,
    arrowstyle="-|>",
    arrowsize=12
)

# Draw labels (ISO3 codes)
nx.draw_networkx_labels(
    G, pos,
    font_size=8,
    font_color="black"
)

plt.title("Global Producer → Manufacturer Dependency Graph")
plt.axis("off")
plt.tight_layout()
plt.show()

# -----------------------------
# EXPORT FOR GEPHI
# -----------------------------
nx.write_gexf(G, PROCESSED_DIR / "dependency_graph.gexf")

print("Gephi export saved as dependency_graph.gexf")
print("Notebook 4 complete.")


In [ ]:
# %%
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from pathlib import Path
import os

PROJECT_ROOT = Path(os.getcwd()).resolve().parents[0]
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

def load_processed(name):
    return pd.read_parquet(PROCESSED_DIR / name)


# -----------------------------
# LOAD NOTEBOOK 3 OUTPUTS
# -----------------------------
links = load_processed("country_links.parquet")
dependency_matrix = load_processed("dependency_matrix.parquet")

# -----------------------------
# REMOVE LINKLESS NODES
# -----------------------------
# A node is meaningful if:
# - it produces minerals
# - OR it manufactures sectors
# - OR it has at least one dependency edge

meaningful_nodes = set()

# Producers
meaningful_nodes.update(
    links.loc[links["Producer Count"] > 0, "Country Code"].tolist()
)

# Manufacturers
meaningful_nodes.update(
    links.loc[links["Manufacturer Count"] > 0, "Country Code"].tolist()
)

# Nodes with edges in dependency matrix
for p in dependency_matrix.index:
    if dependency_matrix.loc[p].sum() > 0:
        meaningful_nodes.add(p)

for m in dependency_matrix.columns:
    if dependency_matrix[m].sum() > 0:
        meaningful_nodes.add(m)

# Filter links table
links = links[links["Country Code"].isin(meaningful_nodes)]

# Filter dependency matrix
dependency_matrix = dependency_matrix.loc[
    dependency_matrix.index.isin(meaningful_nodes),
    dependency_matrix.columns.isin(meaningful_nodes)
]

print(f"Remaining nodes after cleanup: {len(meaningful_nodes)}")

# -----------------------------
# BUILD NETWORKX GRAPH
# -----------------------------
G = nx.DiGraph()

# Add nodes with attributes
for _, row in links.iterrows():
    G.add_node(
        row["Country Code"],
        country=row["Country"],
        producer_strength=row["Producer Strength"],
        manufacturer_strength=row["Manufacturer Strength"],
        capability=row["Capability Score"],
        dependency_class=row["Dependency Class"]
    )

# Add edges
for producer in dependency_matrix.index:
    for manufacturer in dependency_matrix.columns:
        weight = dependency_matrix.loc[producer, manufacturer]
        if weight > 0:
            G.add_edge(producer, manufacturer, weight=weight)

print(f"Graph built: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# -----------------------------
# STATIC NETWORK PLOT
# -----------------------------
plt.figure(figsize=(14, 10))

pos = nx.spring_layout(G, k=0.4, iterations=50)

class_colors = {
    "Integrated": "gold",
    "Producer Only": "steelblue",
    "Manufacturer Only": "firebrick",
    "Neither": "grey"
}

node_colors = [
    class_colors[G.nodes[n]["dependency_class"]] for n in G.nodes()
]

node_sizes = [
    800 * G.nodes[n]["capability"] for n in G.nodes()
]

nx.draw_networkx_nodes(
    G, pos,
    node_color=node_colors,
    node_size=node_sizes,
    alpha=0.9
)

edge_weights = [G[u][v]["weight"] * 5 for u, v in G.edges()]
nx.draw_networkx_edges(
    G, pos,
    width=edge_weights,
    alpha=0.4,
    arrows=True,
    arrowstyle="-|>",
    arrowsize=12
)

nx.draw_networkx_labels(
    G, pos,
    font_size=8,
    font_color="black"
)

plt.title("Global Producer → Manufacturer Dependency Graph (Filtered)")
plt.axis("off")
plt.tight_layout()
plt.show()

# -----------------------------
# EXPORT FOR GEPHI
# -----------------------------
nx.write_gexf(G, PROCESSED_DIR / "dependency_graph_filtered.gexf")

print("Filtered Gephi export saved as dependency_graph_filtered.gexf")
print("Notebook 4 complete.")


In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from pathlib import Path
import os

from utils.utility import load_parquet

PROJECT_ROOT = Path(os.getcwd()).resolve().parents[0]
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

# -------------------------------------------------
# LOAD DATA
# -------------------------------------------------
links = load_parquet("country_links.parquet")
dependency_matrix = load_parquet("dependency_matrix.parquet")
mineral_sector = load_parquet("mineral_sector_mat.parquet")  # rows = sectors, cols = minerals
sector_country = load_parquet("sector_country_matrix.parquet")  # rows = sectors, cols = countries



def clean_iso_index(df):
    df.index = df.index.astype(str).str.strip().str.upper()
    return df

def clean_iso_cols(df):
    df.columns = df.columns.astype(str).str.strip().str.upper()
    return df

# Clean links
links["Country Code"] = links["Country Code"].astype(str).str.strip().str.upper()

# Clean dependency matrix
dependency_matrix = clean_iso_index(dependency_matrix)
dependency_matrix = clean_iso_cols(dependency_matrix)

# Clean mineral-sector matrix
mineral_sector = clean_iso_index(mineral_sector)
mineral_sector = clean_iso_cols(mineral_sector)

# Clean sector-country matrix
sector_country = clean_iso_index(sector_country)
sector_country = clean_iso_cols(sector_country)



# -------------------------------------------------
# GLOBAL GRAPH (unchanged)
# -------------------------------------------------
def build_and_save_graph(G, name):
    plt.figure(figsize=(14, 10))
    pos = nx.spring_layout(G, k=0.4, iterations=50)

    class_colors = {
        "Integrated": "gold",
        "Producer Only": "steelblue",
        "Manufacturer Only": "firebrick",
        "Neither": "grey"
    }

    node_colors = [class_colors[G.nodes[n]["dependency_class"]] for n in G.nodes()]
    node_sizes = [800 * G.nodes[n]["capability"] for n in G.nodes()]
    edge_weights = [G[u][v]["weight"] * 5 for u, v in G.edges()]

    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes, alpha=0.9)
    nx.draw_networkx_edges(G, pos, width=edge_weights, alpha=0.4, arrows=True, arrowstyle="-|>", arrowsize=12)
    nx.draw_networkx_labels(G, pos, font_size=8)

    plt.title(name)
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{name}.png", dpi=300, bbox_inches="tight")
    plt.show()

    nx.write_gexf(G, PROCESSED_DIR / f"{name}.gexf")
    print(f"Saved: {name}")


# -------------------------------------------------
# 1. GLOBAL GRAPH
# -------------------------------------------------
G_global = nx.DiGraph()

for _, row in links.iterrows():
    G_global.add_node(
        row["Country Code"],
        country=row["Country"],
        producer_strength=row["Producer Strength"],
        manufacturer_strength=row["Manufacturer Strength"],
        capability=row["Capability Score"],
        dependency_class=row["Dependency Class"]
    )

for p in dependency_matrix.index:
    for m in dependency_matrix.columns:
        w = dependency_matrix.loc[p, m]
        if w > 0:
            G_global.add_edge(p, m, weight=w)

build_and_save_graph(G_global, "global_dependency_graph")

# -------------------------------------------------
# 2. SECTOR-SPECIFIC GRAPHS (autosave enabled)
# -------------------------------------------------

# Ensure sector indices are clean
mineral_sector.index = mineral_sector.index.str.strip().str.upper()
sector_country.index = sector_country.index.str.strip().str.upper()

# Only build graphs for sectors present in BOTH matrices
sectors = sorted(set(mineral_sector.index) & set(sector_country.index))

for sector in sectors:
    print(f"\n=== Building graph for sector: {sector} ===")

    # Minerals used by this sector
    minerals_used = mineral_sector.loc[sector]
    minerals_used = minerals_used[minerals_used > 0].index.tolist()

    if len(minerals_used) == 0:
        print(f"Skipping {sector}: no minerals used.")
        continue

    # Manufacturers in this sector
    manufacturers = sector_country.loc[sector]
    manufacturers = manufacturers[manufacturers > 0].index.tolist()

    if len(manufacturers) == 0:
        print(f"Skipping {sector}: no manufacturers.")
        continue

    # Producers (countries with any mineral production)
    producers = dependency_matrix.index.tolist()

    minerals_used = [m for m in minerals_used if m in producer_matrix.columns]


    producers = [
    p for p in dependency_matrix.index
    if any(producer_matrix.loc[p, mineral] > 0 for mineral in minerals_used)
]



    # Build sector-specific dependency matrix
    dep_sector = dependency_matrix.loc[
        dependency_matrix.index.isin(producers),
        dependency_matrix.columns.isin(manufacturers)
    ]

    # Build graph
    G = nx.DiGraph()

    # Add nodes
    for _, row in links.iterrows():
        code = row["Country Code"]
        if code in dep_sector.index or code in dep_sector.columns:
            G.add_node(
                code,
                country=row["Country"],
                producer_strength=row["Producer Strength"],
                manufacturer_strength=row["Manufacturer Strength"],
                capability=row["Capability Score"],
                dependency_class=row["Dependency Class"]
            )

    # Add edges
    for p in dep_sector.index:
        for m in dep_sector.columns:
            w = dep_sector.loc[p, m]
            if w > 0:
                G.add_edge(p, m, weight=w)

    if G.number_of_nodes() == 0:
        print(f"Skipping {sector}: empty graph.")
        continue

    # ⭐ AUTOSAVE HERE ⭐
    graph_name = f"dependency_graph_sector_{sector}"
    build_and_save_graph(G, graph_name)



In [ ]:
print(f"Graph built: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
